# Exercise: Transforming Time-Series Data in pandas

You have a clean daily bike-share series. Your job is to compute rolling statistics, test for stationarity, and apply differencing to prepare the series for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
# Clean: deduplicate, fix October 2023 unit error, remove negatives, fill gaps
df = df.groupby("date", as_index=False)["rides"].mean()
df["date"] = pd.to_datetime(df["date"])
mask = (df["date"].dt.year == 2023) & (df["date"].dt.month == 10) & (df["rides"] < 1000)
df.loc[mask, "rides"] = df.loc[mask, "rides"] * 24
df.loc[df["rides"] < 0, "rides"] = np.nan
df = df.set_index("date").asfreq("D")
df["rides"] = df["rides"].interpolate(method="time")
rides = df["rides"]
print(f"Clean series: {len(rides)} days")

In [ ]:
def apply_transforms(df):
    """
    Apply log, rolling mean, and differencing to the "rides" column.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with DatetimeIndex and "rides" column.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with new columns: log_rides, rolling_mean, diff_rides.
    """
    df = df.copy()
    df["log_rides"] = np.log(df["rides"])
    df["rolling_mean"] = df["rides"].rolling(window=...).mean()
    df["diff_rides"] = df["rides"].diff(periods=...)
    return df

df_t = apply_transforms(df)
df_t[["rides", "log_rides", "rolling_mean", "diff_rides"]].head()

def detect_outliers(series, threshold=3.0):
    """
    Flag observations more than threshold standard deviations from the mean.
    
    Parameters
    ----------
    series : pd.Series
    threshold : float
        Number of standard deviations for outlier threshold.
    
    Returns
    -------
    pd.Series
        Boolean series: True where value is an outlier.
    """
    z = (series - series.mean()) / series.std()
    return z.abs() > ...

outliers = detect_outliers(df["rides"])
print(f"Outliers: {outliers.sum()}")
df[outliers]

In [ ]:
def apply_differencing(series):
    """
    Apply first differencing to remove trend.
    
    Parameters
    ----------
    series : pd.Series
    
    Returns
    -------
    pd.Series
        Differenced series with no NaN values.
    """
    return series.diff().dropna()

diffed = apply_differencing(rides)
adf_diff = run_adf_test(diffed)
print(f"Differenced ADF p-value: {adf_diff["p_value"]:.6f}")
print(f"Stationary: {"yes" if adf_diff["p_value"] < 0.05 else "no"}")

def impute_missing(series):
    """
    Forward-fill then backward-fill missing values in a series.
    
    Parameters
    ----------
    series : pd.Series
    
    Returns
    -------
    pd.Series
        Series with no NaN values.
    """
    return series.ffill().bfill()

filled = impute_missing(df["rides"])
print(f"Missing before: {df["rides"].isna().sum()}")
print(f"Missing after: {filled.isna().sum()}")

In [ ]:
def train_test_split(series, test_days=90):
    """
    Split a time series chronologically into train and test sets.
    
    Parameters
    ----------
    series : pd.Series
    test_days : int
        Number of observations to hold out for testing.
    
    Returns
    -------
    tuple[pd.Series, pd.Series]
        (train, test)
    """
    train = series[:-test_days]
    test = series[-test_days:]
    return train, test

train, test = train_test_split(rides)
print(f"Train: {len(train)} days")
print(f"Test: {len(test)} days")

In [ ]:
adf_diff = run_adf_test(diffed)
print(f"Differenced ADF p-value: {adf_diff['p_value']:.6f}")
print(f"Stationary: {'yes' if adf_diff['p_value'] < 0.05 else 'no'}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color=UN["Black"])
axes[0].set_ylabel("Rides")
axes[0].set_title("Original series")
axes[1].plot(diffed.index, diffed.values, color=US["Green"])
axes[1].axhline(y=0, color=UN["Medium Gray"], linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change")
axes[1].set_title("First-differenced series")
plt.tight_layout()
axes[0].set_ylabel("Value", fontsize=16)
for ax in axes:
    ax.tick_params(axis="both", labelsize=14)
plt.show()

## Task 4: Train/Test Split

Split chronologically, holding out the last 90 days for testing.

In [ ]:
def train_test_split(series, test_days=90):
    """
    Split a series into train and test by number of trailing days.
    
    Parameters
    ----------
    series : pd.Series
    test_days : int
    
    Returns
    -------
    tuple[pd.Series, pd.Series]
    """
    train = series.iloc[:-test_days]
    test = series.iloc[-test_days:]
    return train, test

train, test = train_test_split(df["rides"])
print(f"Train: {len(train)} days, Test: {len(test)} days")